# Image Processing Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Brightness rescue.** `cv2.add` saturates instead of wrapping — yet pixels already clipped to an extreme stay lost: saturation is lossy.

In [ ]:
import numpy as np
import cv2

page = np.full((160, 300), 190, dtype=np.uint8)
page[60:100, 40:260] = 60                 # a dark "text" band

dim = cv2.add(page, -70)                  # underexposed scan
rescued = cv2.add(dim, 70)                # shift back up

for name, im in (("page  ", page), ("dim   ", dim), ("fixed ", rescued)):
    print(name, "range:", im.min(), "..", im.max())

print("rescued == page:", bool((rescued == page).all()))
# Nothing wrapped (add clamps instead of wrapping), but the band hit the 0
# floor on the way down and returns as 70, not 60: clipping loses information.

**2. Read a histogram.** Two tall spikes = a bimodal image, exactly where Otsu's automatic cut thrives.

In [ ]:
import numpy as np

doc = np.full((120, 200), 230, dtype=np.uint8)
doc[30:90, 60:140] = 40                    # a dark object

hist, _ = np.histogram(doc, bins=256, range=(0, 256))

print("pixels at 40 :", hist[40])          # 60 * 80 = object area
print("pixels at 230:", hist[230])         # everything else
print("total        :", hist.sum(), "== doc.size =", doc.size)
# Two well-separated spikes -> any threshold between them splits cleanly,
# which is precisely the situation Otsu is designed to solve.

**3. Odd kernels only.** A window needs a true centre pixel to know where its result lands - even sizes have none.

In [ ]:
import numpy as np
import cv2

img = np.full((80, 120), 128, dtype=np.uint8)
img[30:50, 40:80] = 250

try:
    cv2.GaussianBlur(img, (6, 6), 0)                # even size -> error
except cv2.error as e:
    print("cv2.error:", str(e).splitlines()[-1])

smooth = cv2.GaussianBlur(img, (5, 5), 0)           # odd works fine
print("ok:", smooth.shape, smooth.dtype)
# Odd sizes give the window an exact centre pixel to write its answer into.

## Part 2 — Practice

**4. Gamma lookup table.** Precomputing 256 answers turns an expensive power law into one lookup per pixel - gamma < 1 lifts shadows, gamma > 1 sinks them.

In [ ]:
import numpy as np
import cv2

photo = np.full((140, 220), 170, dtype=np.uint8)
photo[40:100, 60:160] = 90
dark = cv2.add(photo, -90)


def gamma_lut(g):
    lut = (np.arange(256, dtype=np.float32) / 255.0) ** g * 255.0
    return np.clip(lut, 0, 255).astype(np.uint8)


fixed = cv2.LUT(dark, gamma_lut(0.45))

print("input 60 via gamma 0.45 ->", gamma_lut(0.45)[60])   # brighter
print("input 60 via gamma 2.00 ->", gamma_lut(2.00)[60])   # darker
print("mean: dark %.1f -> fixed %.1f" % (dark.mean(), fixed.mean()))
# gamma < 1 bends the curve upward: shadows open up without clipping highlights.

**5. Three ways to binarize. `THRESH_BINARY_INV` puts ink at 255 - the polarity OCR expects - and Otsu finds the cut automatically.

In [ ]:
import numpy as np
import cv2

page = np.full((160, 320), 225, dtype=np.uint8)
for i in range(4):
    cv2.putText(page, "scan me", (24, 40 + 34 * i),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, 40, 2, cv2.LINE_AA)

_, hard = cv2.threshold(page, 127, 255, cv2.THRESH_BINARY)
_, inv = cv2.threshold(page, 127, 255, cv2.THRESH_BINARY_INV)
otsu_val, otsu_inv = cv2.threshold(page, 0, 255,
                                   cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

for name, im in (("binary @127", hard), ("inv @127", inv), ("otsu-inv", otsu_inv)):
    print(f"{name:12s} white share: {100 * (im > 0).mean():.1f}%")
print("Otsu picked threshold:", int(otsu_val))
# OCR wants ink = white (255): that's THRESH_BINARY_INV.

**6. Salt, pepper and the right filter.** Averages dissolve the extremes but leave every neighbour still wrong; the median votes each dot out entirely.

In [ ]:
import numpy as np
import cv2

rng = np.random.default_rng(42)
base = np.full((140, 240), 150, dtype=np.uint8)
base[40:100, 70:170] = 90

noisy = base.copy()
sp = rng.random(noisy.shape)
noisy[sp < 0.03] = 0                       # pepper
noisy[sp > 0.97] = 255                     # salt


def report(name, im):
    extremes = int(((im == 0) | (im == 255)).sum())
    err = float(np.abs(im.astype(np.int16) - base.astype(np.int16)).mean())
    print(f"{name}: extreme pixels={extremes:5d}  mean|error|={err:.2f}")


report("noisy    ", noisy)
report("box 5x5  ", cv2.blur(noisy, (5, 5)))
report("gauss 5x5", cv2.GaussianBlur(noisy, (5, 5), 0))
report("median 5 ", cv2.medianBlur(noisy, 5))
# Box/Gaussian push each dot toward its neighbourhood mean: no extremes left,
# but every pixel near it is STILL wrong. The median picks the middle value,
# so the outliers simply lose the vote - lowest error by far.

**7. Unsharp masking.** Adding (sharp - blurred) back restores edge contrast; `std()` is a quick sharpness proxy.

In [ ]:
import numpy as np
import cv2

pattern = np.zeros((160, 240), dtype=np.uint8)
pattern[30:70, 30:110] = 255
pattern[90:130, 130:210] = 255

blurry = cv2.GaussianBlur(pattern, (7, 7), 2.5)     # out-of-focus scanner

kernel = np.array([[0., -1., 0.],
                   [-1., 5., -1.],
                   [0., -1., 0.]])
sharp = cv2.filter2D(blurry, -1, kernel)             # centre weight 5

unsharp = cv2.addWeighted(blurry, 2.0,                        # 1 + amount
                          cv2.GaussianBlur(blurry, (5, 5), 1.5), -1.0, 0)

for name, im in (("crisp    ", pattern), ("blurry   ", blurry),
                 ("kernel-sh", sharp), ("unsharp  ", unsharp)):
    print(f"{name} std = {im.std():.1f}")
# Centre weight 9 sharpens harder - and starts painting halos around edges.

## Part 3 — Challenge

**8. Morphology cleanup crew.** OPEN eats white specks, CLOSE seals black pinholes - both preserve object size - and GRADIENT traces outlines.

In [ ]:
import numpy as np
import cv2

rng = np.random.default_rng(7)
binary = np.zeros((180, 320), dtype=np.uint8)
cv2.rectangle(binary, (30, 40), (120, 140), 255, -1)
cv2.circle(binary, (230, 90), 55, 255, -1)

noise = rng.random(binary.shape)
binary[noise < 0.02] = 255                            # white specks
# sparse pinholes only (~0.5% of the shapes): dense holes would let the
# 7x7 erosion eat the objects themselves
binary[(noise > 0.995) & (binary == 255)] = 0

k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, k)      # erode -> dilate
clean = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, k)      # dilate -> erode
outline = cv2.morphologyEx(clean, cv2.MORPH_GRADIENT, k)  # dilate - erode

for name, im in (("raw           ", binary), ("opened        ", opened),
                 ("opened+closed ", clean)):
    print(f"{name} white pixels: {int((im > 0).sum())}")
print("outline pixels:", int((outline > 0).sum()))
# GRADIENT = dilate - erode, so the outline inherits the kernel's width:
# a 7x7 kernel paints a ~7-pixel-thick band, not a hairline.

**9. The full scan-cleaning pipeline.** Denoise -> binarize -> close -> open: each step assumes the output of the previous one.

In [ ]:
import numpy as np
import cv2

rng = np.random.default_rng(123)

# 0) clean page, then ruin it realistically
clean = np.full((200, 380), 235, dtype=np.uint8)
for i, line in enumerate(["Invoice #0042", "3 x notebooks .. 240 tk",
                          "TOTAL ......... 1250 tk"]):
    cv2.putText(clean, line, (26, 44 + 38 * i),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, 25, 2, cv2.LINE_AA)

gradient = np.tile(np.linspace(210, 175, clean.shape[1]).astype(np.int16),
                   (clean.shape[0], 1))
bad = np.clip(clean.astype(np.int16) - 40 + gradient, 0, 255).astype(np.uint8)
hiss = rng.normal(0, 14, bad.shape).astype(np.int16)
bad = np.clip(bad.astype(np.int16) + hiss, 0, 255).astype(np.uint8)
dust = rng.random(bad.shape)
bad[dust < 0.015] = 0
bad[dust > 0.985] = 255

# 1)-4) the pipeline: denoise, binarize, rejoin, erase
step1 = cv2.medianBlur(bad, 5)
_, step2 = cv2.threshold(step1, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
k3 = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
step3 = cv2.morphologyEx(step2, cv2.MORPH_CLOSE, k3)
step4 = cv2.morphologyEx(step3, cv2.MORPH_OPEN, k3)

print(f"ink coverage: raw binarized {100 * (step2 > 0).mean():.1f}%"
      f" -> cleaned {100 * (step4 > 0).mean():.1f}%")
# Order matters: smoothing before thresholding keeps noise from becoming
# thousands of fake ink pixels; morphology runs only AFTER we are binary.